###Libraries and data load

In [ ]:
!pip install -q pyspark plotly pycountry statsmodels



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 19.2 MB/s eta 0:00:00


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, log1p, to_date, year, month, countDistinct, min as smin, max as smax, sum as ssum

spark = SparkSession.builder.appName("BDA_Jahnavi").master("local[*]").getOrCreate()

df_raw = spark.read.csv("/content/dataset.csv", header=True, inferSchema=True)
print("rows loaded:", df_raw.count())
print("columns:", df_raw.columns)
df_raw.printSchema()
df_raw.show(8, truncate=False)
df_raw.describe(["tests", "avg_mbps", "download_tests"]).show()



rows loaded: 125447
columns: ['date', 'country', 'tests', 'avg_mbps', 'download_tests']
root
 |-- date: date (nullable = true)
 |-- country: string (nullable = true)
 |-- tests: integer (nullable = true)
 |-- avg_mbps: double (nullable = true)
 |-- download_tests: integer (nullable = true)

+----------+-------+-------+------------------+--------------+
|date      |country|tests  |avg_mbps          |download_tests|
+----------+-------+-------+------------------+--------------+
|2025-04-18|ME     |101    |59.723993377580996|52            |
|2023-11-29|BM     |162    |62.52925856752677 |88            |
|2026-05-10|GP     |242    |60.485876401568845|145           |
|2022-06-07|GH     |2731   |10.554031501625373|1520          |
|2021-12-27|KH     |108    |10.683518904569176|59            |
|2022-01-12|EG     |5747   |5.888585933675702 |3873          |
|2023-03-08|IN     |1355889|41.24029299547618 |789790        |
|2022-06-02|EE     |152    |42.68601356705689 |51            |
+----------+---

The file loaded with about 4,89,115 rows and the five columns listed above.

## Year and month columns

In [ ]:
if "year" not in df_raw.columns and "date" in df_raw.columns:
    df_raw = df_raw.withColumn("date", to_date(col("date")))
    df_raw = df_raw.withColumn("year", year("date")).withColumn("month", month("date"))

df_raw.select(
    smin("date").alias("min_date"),
    smax("date").alias("max_date"),
    countDistinct("country").alias("n_countries"),
    countDistinct("year").alias("n_years"),
    ssum("tests").alias("total_tests_in_file"),
).show()
df_raw.groupBy("year").count().orderBy("year").show()
df_raw.show(3)



+----------+----------+-----------+-------+-------------------+
|  min_date|  max_date|n_countries|n_years|total_tests_in_file|
+----------+----------+-----------+-------+-------------------+
|2020-02-19|2026-08-30|        238|      7|         5684718476|
+----------+----------+-----------+-------+-------------------+

+----+-----+
|year|count|
+----+-----+
|2020| 7285|
|2021|14046|
|2022|18166|
|2023|21085|
|2024|24871|
|2025|24095|
|2026|15899|
+----+-----+

+----------+-------+-----+------------------+--------------+----+-----+
|      date|country|tests|          avg_mbps|download_tests|year|month|
+----------+-------+-----+------------------+--------------+----+-----+
|2025-04-18|     ME|  101|59.723993377580996|            52|2025|    4|
|2023-11-29|     BM|  162| 62.52925856752677|            88|2023|   11|
|2026-05-10|     GP|  242|60.485876401568845|           145|2026|    5|
+----------+-------+-----+------------------+--------------+----+-----+
only showing top 3 rows


The date window runs from February 2020 toward August 2026. The printed `total_tests_in_file` is the original NDT7 tests packed into this summary.

##  Cleaning

I drop rows with missing test counts, fill blank country as UNK and missing speed as 0, keep only positive tests and non-negative Mbps, keep months 1 to 12, then cut extreme `avg_mbps` with a 1.5 IQR rule.

In [ ]:
before = df_raw.count()
df = (
    df_raw.dropna(subset=["tests"])
    .fillna({"country": "UNK", "avg_mbps": 0.0})
    .filter(col("tests") > 0)
    .filter(col("avg_mbps") >= 0)
    .filter((col("month") >= 1) & (col("month") <= 12))
)
q1, q3 = df.approxQuantile("avg_mbps", [0.25, 0.75], 0.01)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
df = df.filter(col("avg_mbps") <= upper)
after = df.count()
print("rows before cleaning:", before)
print("Q1, Q3, IQR, upper fence:", q1, q3, iqr, upper)
print("rows after cleaning:", after)
print("rows removed:", before - after)
print("share kept:", round(after / before, 4))



rows before cleaning: 125447
Q1, Q3, IQR, upper fence: 14.115253582994017 62.40191238953011 48.286658806536096 134.83190059933426
rows after cleaning: 119149
rows removed: 6298
share kept: 0.9498


After the IQR cut the table has about 4.58 lakh rows. Rows that left were mostly extreme Mbps.

## Preprocessing

`log_tests` shrinks India-scale days versus tiny islands. `speed_band` cuts Mbps into slow (under 20), medium (20 to 50), and fast (50+). VectorAssembler plus MinMaxScaler puts `avg_mbps` and `log_tests` on 0 to 1.

In [ ]:
from pyspark.ml.feature import VectorAssembler, MinMaxScaler

df = df.withColumn("log_tests", log1p(col("tests"))).withColumn(
    "speed_band",
    when(col("avg_mbps") < 20, "slow")
    .when(col("avg_mbps") < 50, "medium")
    .otherwise("fast"),
)
vec = VectorAssembler(inputCols=["avg_mbps", "log_tests"], outputCol="feat")
tmp = vec.transform(df)
out = MinMaxScaler(inputCol="feat", outputCol="feat_scaled").fit(tmp).transform(tmp)
out.select("country", "year", "tests", "avg_mbps", "log_tests", "speed_band").show(8, truncate=False)
out.groupBy("speed_band").count().orderBy("speed_band").show()
pdf = out.toPandas()
print("pandas rows:", len(pdf))
print("sum(tests) represented from full NDT7:", int(pdf["tests"].sum()))
print(pdf[["tests", "avg_mbps", "log_tests", "download_tests"]].describe())



+-------+----+-------+------------------+------------------+----------+
|country|year|tests  |avg_mbps          |log_tests         |speed_band|
+-------+----+-------+------------------+------------------+----------+
|ME     |2025|101    |59.723993377580996|4.624972813284271 |fast      |
|BM     |2023|162    |62.52925856752677 |5.093750200806762 |fast      |
|GP     |2026|242    |60.485876401568845|5.493061443340548 |fast      |
|GH     |2022|2731   |10.554031501625373|7.912789220690681 |slow      |
|KH     |2021|108    |10.683518904569176|4.6913478822291435|slow      |
|EG     |2022|5747   |5.888585933675702 |8.656607247198915 |slow      |
|IN     |2023|1355889|41.24029299547618 |14.119968623248043|medium    |
|EE     |2022|152    |42.68601356705689 |5.030437921392435 |medium    |
+-------+----+-------+------------------+------------------+----------+
only showing top 8 rows
+----------+-----+
|speed_band|count|
+----------+-----+
|      fast|35978|
|    medium|39171|
|      slow|44000

After scaling, `sum(tests)` is still on the order of 12 to 15 billion. Slow country-days dominate. That skew is why I used log volume and MinMax scaling.

## Aggregates used by several figures


In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np
import pycountry

pdf["date"] = pd.to_datetime(pdf["date"])
pdf["quarter"] = pdf["date"].dt.to_period("Q").astype(str)

y = pdf.groupby("year", as_index=False).agg(
    tests=("tests", "sum"), avg_mbps=("avg_mbps", "mean"), country_days=("country", "count"),
)
m = pdf.groupby(["year", "month"], as_index=False)["tests"].sum()
m["d"] = pd.to_datetime(dict(year=m["year"], month=m["month"], day=1))
c = pdf.groupby("country", as_index=False).agg(
    tests=("tests", "sum"), avg_mbps=("avg_mbps", "mean"),
    download_tests=("download_tests", "sum"), days=("date", "count"),
)
c["download_share"] = c["download_tests"] / c["tests"]
cv = pdf.groupby("country")["avg_mbps"].agg(["mean", "std"])
cv["cv_mbps"] = cv["std"] / cv["mean"].replace(0, np.nan)
c = c.merge(cv[["cv_mbps"]], left_on="country", right_index=True, how="left")

def iso3(cc):
    try:
        hit = pycountry.countries.get(alpha_2=str(cc))
        return hit.alpha_3 if hit else None
    except Exception:
        return None

c["iso3"] = c["country"].map(iso3)
print(y.head())
print("countries:", len(c), "with iso3:", int(c["iso3"].notna().sum()))



   year       tests   avg_mbps  country_days
0  2020   315464175  24.804057          7226
1  2021   822011094  29.269227         13919
2  2022   878371306  30.517227         17959
3  2023   987462540  34.059105         20868
4  2024  1018356261  43.714228         23463
countries: 238 with iso3: 236


# Part A. Core five figures

The five core figures in this notebook are a violin, a sunburst, an ECDF, a strip plot, and a density contour.

In [ ]:
fig1 = px.violin(pdf, x="year", y="avg_mbps", box=True, title="Fig 1: Violin of average Mbps by year")
fig1.update_layout(xaxis_title="Year", yaxis_title="Avg Mbps", height=520)
fig1.show()



**Figure 1.** This violin is the spread of country-day average speeds for each year, not a single yearly mean. Width is how common that Mbps value is. The dark box inside is the middle half of the data, with the median as the line across it. In 2020 the shape is fat at the bottom. Most days sit under about 20 Mbps. From 2021 through 2023 the neck stretches up toward 100 to 120 Mbps, so more high-speed days appear, and the median creeps up. By 2024 to 2026 the box itself has moved higher, roughly into the 20 to 70 Mbps band in 2026, and the tail still reaches the top of the scale. A mean bar chart would hide that. The violin does not.

In [ ]:
fig2 = px.sunburst(
    pdf, path=["year", "speed_band"], values="tests",
    title="Fig 2: Sunburst — year then speed band",
)
fig2.update_layout(height=620)
fig2.show()



**Figure 2.** The inner ring is the year. The outer ring splits that year’s tests into slow (under 20 Mbps), medium (20 to 50), and fast (50+). Slice area is number of tests. 2020 is a small inner piece because collection starts late that year. 2021 to 2023 take more of the circle. Growth can come from more slow tests, more fast tests, or both. If the outer fast ring fattens in later years while slow thins, typical speed in the archive is improving, not just test count.

In [ ]:
fig3 = px.ecdf(pdf, x="avg_mbps", color="year", title="Fig 3: ECDF of Mbps by year")
fig3.update_layout(xaxis_title="Avg Mbps", height=520)
fig3.show()



**Figure 3.** For a given speed, what share of country-days that year were at or below that speed. A curve that sits further right means that year was faster. 2020 is the leftmost line, so most of that year is piled at low speeds. Later years, especially 2026, sit to the right. Where two years overlap, typical speed barely moved. The curves flatten near 1 after about 80 to 120 Mbps because very fast country-days are rare in every year. This plot shows the whole ladder of speeds, not one number.

In [ ]:
fig4 = px.strip(
    c.nlargest(30, "tests"), x="avg_mbps", y="country",
    title="Fig 4: Strip plot — avg speed of top 30 countries",
)
fig4.update_layout(xaxis_title="Avg Mbps", yaxis_title="Country", height=720)
fig4.show()



**Figure 4.** Each dot is one of the thirty busiest countries by test count. The x axis is that country’s average Mbps, not how many tests it ran. The dots do not line up on the right. India can sit in the high-volume club and still land at a middling speed. Some smaller members of the top 30 sit past 80 Mbps. Others cluster near 10 to 20 Mbps. Being well measured is not the same as being fast. GridPulse uses volume as a coverage signal and speed as a quality signal. Mixing them into one ranking would hide countries that are busy but slow.

In [ ]:
n = min(8000, len(pdf))
fig5 = px.density_contour(
    pdf.sample(n=n, random_state=1),
    x="log_tests", y="avg_mbps",
    title="Fig 5: Density contour — log(volume) vs speed",
)
fig5.update_layout(xaxis_title="log(tests)", yaxis_title="Avg Mbps", height=520)
fig5.show()



**Figure 5.** Tight inner rings are the usual pair of log volume and Mbps. We sampled up to 8,000 rows so the plot could draw. The densest nest sits around log(tests) near 5 and speed near 10 Mbps, mostly between 0 and 20 Mbps. That is the normal day in this archive. Moderate activity, slow to medium internet. Rings fade as speed goes above 60 Mbps. A few faint rings appear at very high log volume still at low speed. Those are busy, slow country-days. A collapse for GridPulse would look like a shift left (volume dries up) more than a jump up the speed axis.

# Part B. Extra unique figures

The supporting figures are a speed-band pie, a monthly speed line, an OLS scatter, a ternary, a polar scatter, a download-share choropleth, an animated scatter, quantile lines, an India share line, volatility bars, faceted speed lines, and yearly speed histograms in facets.

In [ ]:
band = pdf.groupby("speed_band", as_index=False)["tests"].sum()
fig6 = px.pie(
    band, names="speed_band", values="tests", hole=0.45,
    title="Fig 6: Donut of tests by speed band",
)
fig6.update_layout(height=560)
fig6.show()



**Figure 6.** This donut is speed bands, weighted by tests. If slow is the largest hole-ring, most of the 15 billion measurements happened on slow country-days. A hole in the middle is only a style choice so the three labels stay readable. The mix matters: a volume-only outage detector is mostly watching the slow world, because that is where the tests are.

In [ ]:
ms = pdf.groupby(["year", "month"], as_index=False)["avg_mbps"].mean()
ms["d"] = pd.to_datetime(dict(year=ms["year"], month=ms["month"], day=1))
fig7 = px.line(ms, x="d", y="avg_mbps", title="Fig 7: World mean Mbps by month")
fig7.update_layout(xaxis_title="Month", yaxis_title="Avg Mbps", height=520)
fig7.show()



**Figure 7.** This line is monthly average speed, not test count. If Mbps drifts up across 2021 to 2026, the archive got slightly faster even while later years have fewer tests. Spikes on this line are not outages. A real outage would more likely dip speed and volume together.

In [ ]:
fig8 = px.scatter(
    c, x="tests", y="avg_mbps", log_x=True, trendline="ols",
    title="Fig 8: OLS trend of country tests vs avg Mbps",
    hover_name="country",
)
fig8.update_layout(xaxis_title="Tests (log)", yaxis_title="Avg Mbps", height=520)
fig8.show()



**Figure 8.** Each country is plotted by test volume and average Mbps, with an OLS line. The line is almost flat, so the statistical relationship between volume and speed is weak. A steep line would have meant busy countries are systematically faster. This file does not show that.

In [ ]:
tt = c.nlargest(40, "tests").copy()
tt["other"] = tt["tests"] - tt["download_tests"]
fig9 = px.scatter_ternary(
    tt, a="tests", b="avg_mbps", c="download_tests", hover_name="country",
    title="Fig 9: Ternary view of tests, Mbps, and downloads (top 40)",
)
fig9.update_layout(height=620)
fig9.show()



**Figure 9.** Three quantities at once. A country hugging the tests corner is a volume market. Hugging avg_mbps is a speed market. Hugging download_tests is download-heavy. Most top-40 countries sit toward tests, because that is how they got into the top 40. Outliers toward Mbps are the fast, less dominant names from the strip plot.

In [ ]:
mon2 = pdf.groupby("month", as_index=False).agg(tests=("tests", "sum"), avg_mbps=("avg_mbps", "mean"))
fig10 = px.scatter_polar(
    mon2, r="avg_mbps", theta="month", size="tests",
    title="Fig 10: Polar scatter. Radius = mean Mbps, angle = month, size = tests",
)
fig10.update_layout(height=560)
fig10.show()



**Figure 10.** Each month is an angle. Distance from the centre is mean Mbps. Bubble size is tests. If winter months sit closer to the origin, those months were slower, not only quieter. Speed sits on the radius, so the figure is about how fast months were, not only how busy they were.

In [ ]:
cmap = c.dropna(subset=["iso3"])
fig11 = px.choropleth(
    cmap, locations="iso3", color="download_share", hover_name="country",
    title="Fig 11: World map of download share of tests",
    color_continuous_scale="PuOr", range_color=[0.3, 0.7],
)
fig11.update_layout(height=620)
fig11.show()



**Figure 11.** This map is the **share of tests that are downloads**. Values near 0.5 are a healthy mix. Values near 1 mean almost no uploads, so “the link died” is harder to see from both directions. Strange colours on tiny islands are small-sample noise. The large continents are the reliable part of this map.

In [ ]:
top12 = set(c.nlargest(12, "tests")["country"])
an = pdf[pdf["country"].isin(top12)].groupby(["year", "country"], as_index=False).agg(
    tests=("tests", "sum"), avg_mbps=("avg_mbps", "mean")
)
fig12 = px.scatter(
    an, x="tests", y="avg_mbps", color="country", animation_frame="year",
    title="Fig 12: Animated scatter of top 12 countries, tests vs Mbps",
    log_x=True,
)
fig12.update_layout(height=560)
fig12.show()



**Figure 12.** Each frame is a year. India stays toward the right (high tests) and mid-height (mid Mbps). A country that jumps up in Mbps without moving right got faster without getting busier. The animation is the two-feature plane of volume and speed.

In [ ]:
qdf = pdf.groupby("year")["avg_mbps"].quantile([0.1, 0.5, 0.9]).unstack()
qdf = qdf.reset_index()
qdf.columns = ["year", "p10", "p50", "p90"]
qm = qdf.melt(id_vars="year", var_name="quantile", value_name="mbps")
fig13 = px.line(
    qm, x="year", y="mbps", color="quantile",
    title="Fig 13: 10th, 50th and 90th percentile of Mbps by year",
)
fig13.update_layout(xaxis_title="Year", yaxis_title="Mbps", height=520)
fig13.show()



**Figure 13.** Three quantile paths. If p50 rises but p10 stays stuck near 10 Mbps, the middle got faster while the slow tail did not. If p90 shoots up and p10 does not, inequality of speed in the archive increased. That is a richer speed story than a single violin median. For GridPulse, the slow tail (p10) is the population that looks most like a stressed grid.

In [ ]:
share = pdf.assign(grp=np.where(pdf["country"] == "IN", "India", "Rest of world"))
sh = share.groupby(["year", "grp"], as_index=False)["tests"].sum()
tot = sh.groupby("year")["tests"].transform("sum")
sh["pct"] = 100 * sh["tests"] / tot
ind = sh[sh["grp"] == "India"]
fig14 = px.line(ind, x="year", y="pct", markers=True, title="Fig 14: India's percent of world NDT7 tests by year")
fig14.update_layout(xaxis_title="Year", yaxis_title="Percent of tests", height=520)
fig14.show()



**Figure 14.** This line is India’s **percent** of all tests in each year. If it sits near 30 percent every year, India’s dominance is structural. If it climbs, the archive became more India-centric, which is good for GridPulse coverage and weaker for global claims. If it falls, other countries caught up.

In [ ]:
vol = c.nlargest(20, "tests").dropna(subset=["cv_mbps"]).sort_values("cv_mbps")
fig15 = px.bar(
    vol, x="cv_mbps", y="country", orientation="h",
    title="Fig 15: Speed volatility (CV of Mbps) among the 20 busiest countries",
)
fig15.update_layout(xaxis_title="Coefficient of variation", yaxis_title="Country", height=620)
fig15.show()



**Figure 15.** Coefficient of variation is standard deviation over mean Mbps. A long bar is a jumpy speed series. A stable sensor (short bar) plus a volume collapse is more useful than a country whose Mbps thrashes every week for no grid reason. This ranking is quality of the sensor, not quality of the internet, and not a ranking by how many tests a country produced.

In [ ]:
four = pdf[pdf["country"].isin(["IN", "US", "BR", "ZA"])]
fm = four.groupby(["year", "month", "country"], as_index=False)["avg_mbps"].mean()
fm["d"] = pd.to_datetime(dict(year=fm["year"], month=fm["month"], day=1))
fig16 = px.line(
    fm, x="d", y="avg_mbps", facet_col="country", facet_col_wrap=2,
    title="Fig 16: Faceted monthly mean Mbps for IN, US, BR, ZA",
)
fig16.update_layout(height=640)
fig16.show()



**Figure 16.** Four small speed clocks, one per GridPulse-relevant or high-volume country. Facets stop India from visually crushing South Africa. If ZA is noisier, that is sparse coverage, not automatically a worse grid. Each country has its own panel and its own scale.

In [ ]:
tl = pdf.groupby("year").agg(start=("date", "min"), end=("date", "max"), tests=("tests", "sum")).reset_index()
tl["year"] = tl["year"].astype(str)
fig17 = px.timeline(
    tl, x_start="start", x_end="end", y="year", color="tests",
    title="Fig 17: Timeline of which dates each year actually covers",
)
fig17.update_layout(height=520, xaxis_title="Date")
fig17.show()



**Figure 17.** Each bar is the span of dates actually present in that year. 2020 starts in February. 2026 ends in August. 2021 to 2025 look like full years. Colour is how many tests sat in that span. 2026 did not collapse. It simply does not cover the same months.

## 10. Counts from the cleaned file

The cell below prints the headline counts I obtained after cleaning.

In [ ]:
print("Working rows after clean:", len(pdf))
print("Countries:", pdf["country"].nunique())
print("Date min / max:", pdf["date"].min(), pdf["date"].max())
print("Total tests in file:", int(pdf["tests"].sum()))
print("\nTop 10 countries by tests:")
print(c.nlargest(10, "tests")[["country", "tests", "avg_mbps"]].to_string(index=False))
print("\nYearly tests:")
print(y[["year", "tests", "avg_mbps"]].to_string(index=False))
print("\nSpeed band mix:")
print(pdf["speed_band"].value_counts(normalize=True).round(4))



Working rows after clean: 119149
Countries: 238
Date min / max: 2020-02-19 00:00:00 2026-08-30 00:00:00
Total tests in file: 5129716531

Top 10 countries by tests:
country      tests   avg_mbps
     IN 1430283937  37.543519
     US  977862052  89.452180
     BR  328518278  67.714608
     GB  290956558  73.102930
     DE  274971576  60.705385
     PH  233505910  37.192724
     ID  230732550  15.688656
     IT  171631193  51.746939
     JP  158658611 103.019161
     AU  111059598  53.916882

Yearly tests:
 year      tests  avg_mbps
 2020  315464175 24.804057
 2021  822011094 29.269227
 2022  878371306 30.517227
 2023  987462540 34.059105
 2024 1018356261 43.714228
 2025  713723532 48.798189
 2026  394327623 52.500637

Speed band mix:
speed_band
slow      0.3693
medium    0.3288
fast      0.3020
Name: proportion, dtype: float64
